# Du graphe causal au do-calculus — le pont entre les quatre séries causales

> **Notebook-pont** de la constellation causale du dépôt (règle F : `dowhy` installé et exécuté réellement, SOTA-OK). Il ne remplace pas les notebooks dédiés de chaque moteur : il donne l'**armature formelle unifiée** (échelle de Pearl + trois règles du do-calculus) et la fait tourner sur l'**outil de référence** [`dowhy`](https://www.pywhy.org/dowhy/), avant de renvoyer à chaque série pour l'instanciation par moteur.

La causalité est traitée à **quatre endroits** du dépôt, chacun avec son moteur et son angle :

| Série | Moteur | Angle |
|-------|--------|-------|
| [Tweety-11](../../../SymbolicAI/Tweety/Tweety-11-Causal.ipynb) | Tweety (.NET, logique) | SCM, opérateur `do`, contrefactuels |
| [Infer-14](../../Infer/Infer-14-Causal-Inference.ipynb) | Infer.NET (message passing) | backdoor, front-door, Simpson, médiation |
| [PyMC-14](../../PyMC/PyMC-14-Causal-Inference.ipynb) | PyMC (MCMC) | backdoor, front-door, contrefactuel bayésien |
| [ICT-5](../../../IIT/ICT-Series/ICT-5-CausalEmergence.ipynb) / [ICT-6](../../../IIT/ICT-Series/ICT-6-SortingToTPM-CausalEmergence.ipynb) | PyPhi (CE 2.0) | **émergence causale** (information effective de Hoel) |

Ce que ce pont ajoute, que les quatre notebooks ne font pas chacun séparément : (1) la **théorie unifiée** du do-calculus présentée une bonne fois ; (2) une exécution sur l'**outil SOTA** `dowhy` qui **identifie** l'estimande (backdoor/front-door/IV), **estime** puis **réfute** ; (3) la **mise en regard** des deux grands paradigmes — l'interventionnisme de Pearl et l'émergence causale de Hoel.

## Objectifs d'apprentissage

À l'issue de ce notebook vous saurez :

1. **Situer** chaque série causale du dépôt sur l'échelle de Pearl (observation / intervention / contrefactuel).
2. **Énoncer** les trois règles du do-calculus et les lire comme des chirurgies du graphe causal.
3. **Reconnaître** les critères *backdoor* et *front-door*, et les **exécuter** avec `dowhy` sur des données simulées à effet connu.
4. **Distinguer** la causalité interventionniste (Pearl) de l'émergence causale (Hoel / information effective) — deux réponses différentes à la question « quelle échelle cause ? ».

**Prérequis** : probabilités conditionnelles, graphes orientés acycliques (DAG), notions d'inférence bayésienne. Une lecture préalable de l'un des quatre notebooks ci-dessus rend le pont plus concret.

## 1. L'échelle de Pearl — trois niveaux de causalité

Judea Pearl structure la causalité en **trois échelons** croissants, chacun subsumant le précédent :

| Niveau | Question type | Symbole | Opération |
|:------:|---------------|---------|-----------|
| **1. Association** | *Que vois-je ?* | $P(y \mid x)$ | observer |
| **2. Intervention** | *Que se passe-t-il si je fais $x$ ?* | $P(y \mid do(x))$ | agir |
| **3. Contrefactuel** | *Que se serait-il passé si j'avais fait $x'$ ?* | $P(y_x \mid x', y')$ | imaginer |

Le saut du niveau 1 au niveau 2 est **le** cœur du do-calculus : $P(y \mid x)$ (ce qu'on observe dans les données) diffère en général de $P(y \mid do(x))$ (l'effet d'une intervention), précisément à cause des **chemins confondants** que le do-calculus permet de neutraliser. Les quatre notebooks de la constellation couvrent les trois niveaux depuis l'angle de leur moteur ; nous récapitulons ici le **formalisme** partagé.

## 2. Les trois règles du do-calculus

Soit $G$ un graphe causal, $G_{\overline{Z}}$ (resp. $G_{\underline{Z}}$) le graphe où l'on **coupe** les arêtes entrant dans (resp. sortant de) $Z$. Les trois règles de Pearl permettent de transformer toute expression avec $do(\cdot)$ en expression sans $do(\cdot)$ lorsque c'est possible :

> **Règle 1 (insertion/suppression d'observations).**
> $P(y \mid do(x), z, w) = P(y \mid do(x), w)$ si $(Y \perp\!\!\!\perp Z \mid X, W)_{G_{\overline{X}}}$.

> **Règle 2 (échange action/observation).**
> $P(y \mid do(x), do(z), w) = P(y \mid do(x), z, w)$ si $(Y \perp\!\!\!\perp Z \mid X, W)_{G_{\overline{X}\underline{Z}}}$.

> **Règle 3 (insertion/suppression d'actions).**
> $P(y \mid do(x), do(z), w) = P(y \mid do(x), w)$ si $(Y \perp\!\!\!\perp Z \mid X, W)_{G_{\overline{X}\overline{Z(W)}}}$.

**Intuition unifiée** : chaque règle est une **chirurgie du graphe** suivie d'un test de $d$-séparation. Si deux variables sont indépendantes conditionnellement dans le graphe mutilé approprié, on peut réécrire la probabilité. Le critère *backdoor* (§4) et le critère *front-door* (§5) sont les **cas spéciaux les plus utiles** qui en découlent directement.

## 3. Configuration — l'outil de référence `dowhy`

Nous utilisons [`dowhy`](https://www.pywhy.org/dowhy/) (écosystème PyWhy), la librairie de référence pour l'inférence causale. Elle poursuit le pipeline en **quatre étapes** qui structurent toute analyse causale sérieuse :

1. **Modèle** — on fournit les données **et** le graphe causal (hypothèses expertes) ;
2. **Identification** — `dowhy` lit le graphe et détermine *quel* estimande est calculable (backdoor, front-door, variable instrumentale) ;
3. **Estimation** — on choisit un estimateur (régression, appariement, pondération inverse) ;
4. **Réfutation** — on stress-teste l'estimation (placebo, sous-échantillon, proxy) pour vérifier sa robustesse.

C'est précisément la rigueur (identification **avant** estimation) qui distingue une analyse causale d'une simple régression corrélationnelle.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=UserWarning, module="dowhy")
import numpy as np
import pandas as pd
import networkx as nx
from dowhy import CausalModel
import dowhy

print("dowhy", dowhy.__version__, "| networkx", nx.__version__, "| pandas", pd.__version__)

dowhy 0.14 | networkx 3.6.1 | pandas 3.0.2


## 4. Le critère *backdoor* — neutraliser le confondeur observable

**Énoncé** (Pearl). Un ensemble $Z$ satisfait le critère *backdoor* relativement à $(X, Y)$ si :

1. aucun nœud de $Z$ n'est un descendant de $X$ ;
2. $Z$ **bloque tous les chemins backdoor** (chemins entrant dans $X$ par une arête pointée vers $X$) entre $X$ et $Y$.

Quand un tel $Z$ existe et est **observable**, alors

$$P(y \mid do(x)) = \sum_z P(y \mid x, z)\, P(z),$$

ce qui ramène l'intervention à un ajustement — **règle 2 du do-calculus**.

### Exemple : aptitude → {études, salaire}

Le facteur d'**aptitude** $U$ cause à la fois la décision de **faire des études** ($T$) et le **salaire** $Y$. L'effet vrai de $T$ sur $Y$ est $2{,}0$. La simple comparaison des salaires moyens est **biaisée** car $U$ ouvre un chemin backdoor $T \leftarrow U \rightarrow Y$.

In [2]:
np.random.seed(42)
N = 20_000
aptitude = np.random.normal(0, 1, N)
college  = (0.8 * aptitude + np.random.normal(0, 1, N) > 0).astype(int)
earnings = 2.0 * college + 1.5 * aptitude + np.random.normal(0, 1, N)
df_backdoor = pd.DataFrame({"aptitude": aptitude, "college": college, "earnings": earnings})

naive = df_backdoor.query("college == 1").earnings.mean() - df_backdoor.query("college == 0").earnings.mean()
print(f"Effet NAIF observé : {naive:.3f}   (biaisé — l'aptitude confond)")
print(f"Effet VRAI (connu) : 2.000")

Effet NAIF observé : 3.494   (biaisé — l'aptitude confond)
Effet VRAI (connu) : 2.000


In [3]:
# Graphe causal : aptitude -> college, college -> earnings, aptitude -> earnings
gml_backdoor = '''
graph [
  directed 1
  node [ id 0 label "aptitude" ]
  node [ id 1 label "college" ]
  node [ id 2 label "earnings" ]
  edge [ source 0 target 1 ]
  edge [ source 1 target 2 ]
  edge [ source 0 target 2 ]
]
'''
model_bd = CausalModel(data=df_backdoor, treatment="college", outcome="earnings", graph=gml_backdoor)
estimand_bd = model_bd.identify_effect(proceed_when_unidentifiable=False)
print(estimand_bd)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
    d                           
──────────(E[earnings|aptitude])
d[college]                      
Estimand assumption 1, Unconfoundedness: If U→{college} and U→earnings then P(earnings|college,aptitude,U) = P(earnings|college,aptitude)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
    d                           
──────────(E[earnings|aptitude])
d[college]                      
Estimand assumption 1, Unconfoundedness: If U→{college} and U→earnings then P(earnings|college,aptitude,U) = P(earnings|college,aptitude)



In [4]:
estimate_bd = model_bd.estimate_effect(estimand_bd, method_name="backdoor.linear_regression")
print(f"Effet estime (dowhy, ajustement backdoor) : {estimate_bd.value:.3f}")
print(f"Effet vrai                                : 2.000")

# Réfutation : l'effet résiste-t-il à un placebo et à un sous-échantillon ?
refute_placebo = model_bd.refute_estimate(estimand_bd, estimate_bd, method_name="placebo_treatment_refuter")
refute_subset  = model_bd.refute_estimate(estimand_bd, estimate_bd, method_name="data_subset_refuter")
print("\nPlacebo (effet attendu ~0) :", round(refute_placebo.new_effect, 3))
print("Sous-echantillon           :", round(refute_subset.new_effect, 3))

Effet estime (dowhy, ajustement backdoor) : 1.981
Effet vrai                                : 2.000



Placebo (effet attendu ~0) : -0.001
Sous-echantillon           : 1.981


### Interprétation

- L'effet **naïf** (~3,49) **surévalue** l'effet des études : les diplômés gagnent plus en partie parce qu'ils étaient **plus aptes** au départ (chemin backdoor).
- Une fois **ajusté sur l'aptitude** (l'ensemble backdoor $\{U\}$), `dowhy` recouvre ~1,98 ≈ effet vrai 2,0 : la chirurgie graphique a neutralisé le confondeur.
- Les **réfutations** confirment la robustesse : l'effet tombe vers 0 sous placebo (un faux traitement ne donne rien) et reste stable sur un sous-échantillon.

## 5. Le critère *front-door* — quand le confondeur est inobservable

Que faire si le confondeur est **inobservable** (génétique, préférences latentes) ? Si l'on dispose d'un **médiateur** $M$ captant tout l'effet de $X$ sur $Y$, le critère *front-door* sauve la situation :

1. $M$ **intercepte** tous les chemins dirigés de $X$ vers $Y$ ;
2. il n'existe pas de chemin backdoor de $X$ vers $M$ ;
3. tous les chemins backdoor de $M$ vers $Y$ sont bloqués par $X$.

Alors $P(y \mid do(x)) = \sum_m P(m \mid x) \sum_{x'} P(y \mid x', m)\, P(x')$.

### Exemple : génotype (latent) → tabagisme → goudron → cancer

Un **génotype** $U$ (non observé) pousse à la fois au **tabagisme** $X$ et au **cancer** $Y$. Impossible d'ajuster sur $U$ : on n'a pas la donnée. Mais le tabagisme n'affecte le cancer **qu'à travers le goudron** $M$, qu'on mesure. `dowhy` doit alors **échouer** à trouver une backdoor et **basculer** sur la front-door via le médiateur.

In [5]:
np.random.seed(7)
N = 20_000
genotype = np.random.normal(0, 1, N)                      # CONFOUNDEUR LATENT (non observé)
smoking  = (1.0 * genotype + np.random.normal(0, 1, N) > 0).astype(int)
tar      = 0.9 * smoking + np.random.normal(0, 0.5, N)
cancer   = 1.0 * tar + 0.8 * genotype + np.random.normal(0, 1, N)
# On n'observe QUE smoking, tar, cancer (le génotype reste caché)
df_frontdoor = pd.DataFrame({"smoking": smoking, "tar": tar, "cancer": cancer})

In [6]:
# Graphe : U(latent) -> smoking, smoking -> tar, tar -> cancer, U(latent) -> cancer
gml_frontdoor = '''
graph [
  directed 1
  node [ id 0 label "U" ]
  node [ id 1 label "smoking" ]
  node [ id 2 label "tar" ]
  node [ id 3 label "cancer" ]
  edge [ source 0 target 1 ]
  edge [ source 1 target 2 ]
  edge [ source 2 target 3 ]
  edge [ source 0 target 3 ]
]
'''
model_fd = CausalModel(data=df_frontdoor, treatment="smoking", outcome="cancer", graph=gml_frontdoor)
estimand_fd = model_fd.identify_effect(proceed_when_unidentifiable=True)
print(estimand_fd)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
No such variable(s) found!

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
Estimand expression:
 ⎡  d                d            ⎤
E⎢──────(cancer)⋅──────────([tar])⎥
 ⎣d[tar]         d[smoking]       ⎦
Estimand assumption 1, Full-mediation: tar intercepts (blocks) all directed paths from smoking to c,a,n,c,e,r.
Estimand assumption 2, First-stage-unconfoundedness: If U→{smoking} and U→{tar} then P(tar|smoking,U) = P(tar|smoking)
Estimand assumption 3, Second-stage-unconfoundedness: If U→{tar} and U→cancer then P(cancer|tar, smoking, U) = P(cancer|tar, smoking)

### Estimand : 4
Estimand name: general_adjustment
No such variable(s) found!



In [7]:
estimate_fd = model_fd.estimate_effect(estimand_fd, method_name="frontdoor.two_stage_regression")
print(f"Effet estime (front-door, deux etapes) : {estimate_fd.value:.3f}")
print(f"Effet vrai via le mediateur             : ~0.900  (0.9 * 1.0)")

Effet estime (front-door, deux etapes) : 0.934
Effet vrai via le mediateur             : ~0.900  (0.9 * 1.0)


### Interprétation

`dowhy` confirme le raisonnement :

- **backdoor** : *« No such variable(s) found! »* — $U$ est latent, aucun ajustement direct possible ;
- **front-door** : l'estimande identifiée passe par le médiateur `tar` ($E[\frac{\partial\, cancer}{\partial\, tar} \cdot \frac{\partial\, tar}{\partial\, smoking}]$) ;
- l'estimation (~0,93) recouvre l'effet vrai (~0,9) **sans jamais observer le génotype**. C'est toute la puissance du front-door : neutraliser un confondeur latent via un médiateur mesurable.

## 6. Comment chaque moteur instancie le do-calculus

Les quatre séries **implantent le même formalisme** (§1-2) avec des outils différents. Partout, $do(X=x)$ se traduit par une **mutilation du graphe** : on supprime les arêtes entrant dans $X$ (déconnexion du mécanisme générant $X$) puis on fige $X=x$.

| Série | Moteur | `do(X)` opérationnellement | Angle couvert |
|-------|--------|----------------------------|---------------|
| [Infer-14](../../Infer/Infer-14-Causal-Inference.ipynb) | Infer.NET | mutilation + **message passing exact** | backdoor, front-door, Simpson, médiation |
| [PyMC-14](../../PyMC/PyMC-14-Causal-Inference.ipynb) | PyMC | mutilation + **échantillonnage MCMC** | backdoor, front-door, contrefactuel bayésien |
| [Tweety-11](../../../SymbolicAI/Tweety/Tweety-11-Causal.ipynb) | Tweety (.NET) | **backend causal logique**, opérateur `do` natif | SCM, contrefactuels |
| **Ce pont** | `dowhy` | **identification + estimation + réfutation** | formalisme unifié + outil SOTA |

Le présent notebook est **complémentaire** : là où Infer-14 et PyMC-14 instrumentent le `do` à la main sur leur moteur, `dowhy` **automatise l'identification** (lit le graphe, choisit backdoor/front-door/IV) puis **estime et réfute** — le pipeline qu'on utiliserait en pratique pour une vraie étude causale.

## 7. Deux paradigmes : Pearl (intervention) vs Hoel (émergence causale)

Les notebooks [ICT-5](../../../IIT/ICT-Series/ICT-5-CausalEmergence.ipynb) et [ICT-6](../../../IIT/ICT-Series/ICT-6-SortingToTPM-CausalEmergence.ipynb) défendent une thèse **différente** et complémentaire : la **causalité émergente** d'Erik Hoel. Là où Pearl demande « *quel est l'effet d'une intervention sur $X$ ?* » au sein d'un graphe **fixé**, Hoel demande « **quelle échelle** de description du système porte le plus de causalité ? ».

Le formalisme de Hoel repose sur l'**information effective** (EI), mesurée sur la matrice de transition d'un système :

$$\text{EI} = \underbrace{\text{déterminisme}}_{\text{le futur est-il prévisible ?}}
            \;-\; \underbrace{\text{dégénérescence}}_{\text{plusieurs passés} \to \text{même futur}}.$$

Un **coarse-graining** (regroupement de micro-états en macro-état) peut **augmenter** l'EI : la macro-échelle est alors *plus* causale que la micro. ICT-5 le démontre sur le réseau canonique de PyPhi (EI passe de ~0,11 à ~0,60).

|  | **Pearl** (do-calculus) | **Hoel** (émergence causale) |
|---|---|---|
| Question | Effet d'une intervention ? | Quelle échelle cause le plus ? |
| Objet | un graphe causal **fixé** | l'**échelle de description** optimale |
| Outil | `dowhy`, Infer.NET, PyMC | PyPhi (CE 2.0) |
| Causalité vue comme | chirurgie du graphe ($do$) | information effective (EI) |

**Pont conceptuel** : les deux cadres répondent à « où est la causalité ? » — Pearl la localise dans les **flèches** qu'on coupe, Hoel dans l'**échelle** qui maximise l'information sur le futur. Un système peut admettre une description causale riche au sens de Hoel (forte EI macro) tout en se prêtant au do-calculus de Pearl au niveau micro : ce sont deux **loupes** différentes, non concurrentes.

## 8. Exercices

Les exercices suivent la convention du dépôt (stub à compléter). Aucun ne lève d'erreur : remplacez `None` par votre code.

### Exercice 1 — Ajouter un second confondeur à la backdoor

Ajoutez un confondeur **observable** `motivation` (qui cause `college` **et** `earnings`) au modèle backdoor, puis demandez à `dowhy` d'identifier l'estimande. L'ensemble backdoor doit maintenant contenir `{aptitude, motivation}`.

### Exercice 2 — Rompre le critère front-door

Modifiez le graphe front-door en ajoutant un chemin **direct** `smoking -> cancer` (contournant le médiateur `tar`). Vérifiez que le critère front-door **échoue** à identifier l'effet — l'hypothèse de médiation complète est rompue.

### Exercice 3 — Comparer effet naïf et effet ajusté sur un nouveau jeu

Générez un nouveau jeu backdoor avec un effet vrai de **5,0** et un confondeur plus fort (coefficient 2,5 sur l'aptitude). Calculez l'effet naïf puis l'effet ajusté, et **commentez** l'écart relatif.

In [8]:
# Exercice 1 — ajoutez un 2e confondeur observable au modèle backdoor.
# Objectif : dowhy doit identifier l'ensemble backdoor {aptitude, motivation}.
# Indice : motivation -> college ET motivation -> earnings (mêmes flèches que aptitude).
# TODO etudiant : construisez df_ex1, le graphe gml_ex1, puis identifiez l'estimande.
df_ex1 = None        # TODO etudiant
gml_ex1 = None       # TODO etudiant
estimand_ex1 = None  # TODO etudiant
print("Exercice 1 a completer")

Exercice 1 a completer


In [9]:
# Exercice 2 — ajoutez un chemin direct smoking -> cancer (médiation rompue).
# Objectif : montrer que l'identification front-door échoue.
# Indice : ajoutez 'edge [ source 1 target 3 ]' au graphe gml_frontdoor.
# TODO etudiant : définissez gml_ex2 avec ce chemin direct, identifiez l'effet.
gml_ex2 = None  # TODO etudiant
print("Exercice 2 a completer")

Exercice 2 a completer


In [10]:
# Exercice 3 — nouvel effet vrai = 5.0, confondeur plus fort (2.5*aptitude).
# Objectif : comparer effet naïf vs ajusté, commenter l'écart relatif.
# TODO etudiant : générez les données, calculez naive puis estimate_bd, affichez l'écart.
print("Exercice 3 a completer")

Exercice 3 a completer


## 9. Synthèse

Vous avez parcouru l'**armature formelle** partagée par les quatre séries causales du dépôt :

- **Échelle de Pearl** (§1) : observation < intervention < contrefactuel — le do-calculus opère le saut $P(y\mid x) \to P(y\mid do(x))$.
- **Trois règles** (§2) : chacune est une chirurgie du graphe + un test de $d$-séparation.
- **Backdoor** (§4) : ajustement sur les confondeurs observables — exécuté par `dowhy`, recouvre l'effet vrai.
- **Front-door** (§5) : sauvetage par médiateur quand le confondeur est latent.
- **Pearl vs Hoel** (§7) : intervention sur un graphe fixé *vs* échelle de description optimale (information effective).

### Constellation causale — où approfondir

| Direction | Notebook | Ce qu'il apporte de plus |
|-----------|----------|--------------------------|
| Logique + SCM + contrefactuels | [Tweety-11](../../../SymbolicAI/Tweety/Tweety-11-Causal.ipynb) | backend causal .NET, opérateur `do` natif |
| Message passing exact | [Infer-14](../../Infer/Infer-14-Causal-Inference.ipynb) | Simpson, médiation, capstone contrefactuel |
| MCMC bayésien | [PyMC-14](../../PyMC/PyMC-14-Causal-Inference.ipynb) | incertitude postérieure sur l'effet |
| Émergence causale (Hoel) | [ICT-5](../../../IIT/ICT-Series/ICT-5-CausalEmergence.ipynb), [ICT-6](../../../IIT/ICT-Series/ICT-6-SortingToTPM-CausalEmergence.ipynb) | information effective, coarse-graining |

> **Pont suivant** (après *merge* de ce notebook) : câbler les quatre notebooks en aller-retour vers ce pont, pour que chaque série renvoie ici pour le formalisme unifié.